# Activity 3 Capstone: End-to-End Pipeline
### Day 2: Data Cleaning & Preparation (SDA ML & Data Science Bootcamp)

**Time:** ~45-50 minutes (this is intentionally the least scaffolded activity — budget generously)

**Dataset:** Telco Customer Churn (7,043 customers, real telecom data)

**Task type:** Classification (churn / no churn) — a deliberate change from Activities 1-2's regression task

**Maps to:** Everything from Day 2, applied independently, end-to-end

---
### The brief

You're a data professional at a telecom company. Leadership wants a model that flags customers likely to **churn** (cancel service) so the retention team can reach out first.

Your job in this notebook is **not** to build the best possible model it's to build a **clean, leakage-free pipeline** that gets a dataset from raw to model-ready, the same way you did in Activities 1 and 2. This time, you'll do it with much less hand-holding, and on a dataset with different quirks:

- A hidden missing-data trap
- Class imbalance
- A deliberate **leaky vs. correct** comparison, so you can *see* the cost of getting this wrong both a subtle version and a catastrophic one

Sections are numbered like a checklist. Code cells have light scaffolding you're expected to write most of the logic yourself. A **check** cell follows each major section so you know you're on track before moving on, and a collapsed **Solution** immediately follows that if you get stuck same pattern as Activities 1 and 2, just with lighter scaffolding in the TODOs themselves.

> The check cells assume you name your variables `X_train`, `X_test`, `y_train`, `y_test` (standard scikit-learn convention) stick with these names and the checks will work automatically.


## 1. Setup & Load

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

pd.set_option("display.max_columns", 25)
sns.set_style("whitegrid")

PRIMARY_URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
BACKUP_URL  = "https://raw.githubusercontent.com/treselle-systems/customer_churn_analysis/master/WA_Fn-UseC_-Telco-Customer-Churn.csv"

try:
    df = pd.read_csv(PRIMARY_URL)
    print("Loaded from primary source.")
except Exception as e:
    print(f"Primary source failed ({e}). Trying backup mirror...")
    try:
        df = pd.read_csv(BACKUP_URL)
        print("Loaded from backup mirror.")
    except Exception as e2:
        print(f"Backup also failed ({e2}).")
        print("Both sources are unreachable -- likely a network/firewall issue.")
        print("Ask your instructor for the CSV file, then run:")
        print('  from google.colab import files')
        print('  uploaded = files.upload()  # choose the CSV file')
        print('  df = pd.read_csv(list(uploaded.keys())[0])')

print(df.shape)
df.head()


Loaded from primary source.
(7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 2. Diagnose

Run your standard audit: shape, dtypes, `isnull().sum()`, duplicate check, target balance.

**Task:** Look closely at `TotalCharges`. Its dtype and its `isnull()` count don't tell the full story dig one level deeper before you trust it.

In [3]:
df.info()
print()
print("Missing values reported by isnull():")
print(df.isnull().sum().sum())
print()
print("Churn balance:")
print(df["Churn"].value_counts(normalize=True))

# Your investigation of TotalCharges here:

# Check hidden missing values in TotalCharges
total_charges_numeric = pd.to_numeric(df["TotalCharges"], errors="coerce")
hidden_nas = total_charges_numeric.isnull().sum()
print("Hidden NaNs in TotalCharges:", hidden_nas)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


> Hint: try `pd.to_numeric(df["TotalCharges"], errors="coerce")` and compare how many values become `NaN` versus what `isnull()` told you originally. This is the "text string hiding in a numeric column" problem from lecture — it won't show up until you try to actually use the column as a number.

<details>
<summary> Solution</summary>

```python
coerced = pd.to_numeric(df["TotalCharges"], errors="coerce")
print("Nulls after coercion:", coerced.isnull().sum())   # reveals the hidden blanks
```
</details>


## 3. Clean

Based on what you found:
- Convert `TotalCharges` to numeric properly, then decide: impute or drop the handful of resulting nulls?
- Drop the `customerID` column it's a unique identifier with no predictive value, and one-hot encoding it would create thousands of useless columns
- Check for exact duplicate rows

In [4]:
# Your cleaning code here
# Clean the dataset based on requirements
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df = df.dropna(subset=["TotalCharges"])

if "customerID" in df.columns:
  df = df.drop(columns=["customerID"])

df = df.drop_duplicates()

In [5]:
# Check your progress
assert "customerID" not in df.columns, "customerID should be dropped."
assert df["TotalCharges"].dtype != object, "TotalCharges should be numeric by now -- did you pd.to_numeric() it?"
assert df["TotalCharges"].isnull().sum() == 0, "TotalCharges still has nulls -- impute or drop them."
assert df.duplicated().sum() == 0, "Duplicate rows remain -- did you call .drop_duplicates()?"
print("Section 3 looks good ✅")


Section 3 looks good ✅


<details>
<summary>Solution</summary>

```python
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].median())
df = df.drop(columns=["customerID"])
df = df.drop_duplicates()
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})
```
</details>


## 4. Split - Before Anything Else

This dataset is **imbalanced** (~73% / 27%). Use `stratify=` so both splits preserve that ratio.

Target column: `Churn` (map `"Yes"`/`"No"` to `1`/`0` first, if you haven't already in Section 3).

In [6]:
# Your split code here (remember: stratify=y)
# Churn to 1 and 0
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

# Define features (X) and target (y)
X = df.drop(columns=["Churn"])
y = df["Churn"]

# Split the data with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [7]:
# Check your progress
assert set(df["Churn"].unique()) <= {0, 1}, "Churn should be mapped to 0/1 by now."
assert "X_train" in dir() and "X_test" in dir(), "Did you create X_train/X_test?"
train_rate = y_train.mean()
test_rate = y_test.mean()
assert abs(train_rate - test_rate) < 0.03, \
    f"Train churn rate ({train_rate:.3f}) and test churn rate ({test_rate:.3f}) differ more than expected -- did you use stratify=y?"
print(f"Train churn rate: {train_rate:.3f}  Test churn rate: {test_rate:.3f} stratification looks correct")


Train churn rate: 0.265  Test churn rate: 0.265 stratification looks correct


<details>
<summary>Solution</summary>

```python
X = df.drop(columns=["Churn"])
y = df["Churn"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
```
</details>


## 5. Encode & Scale Fit on Train Only

- Binary Yes/No columns (`Partner`, `Dependents`, `PhoneService`, `PaperlessBilling`, etc.) → map to 0/1
- Multi-category columns (`InternetService`, `Contract`, `PaymentMethod`, etc.) → One-Hot Encode
- Numeric columns (`tenure`, `MonthlyCharges`, `TotalCharges`) → scale with `StandardScaler`

Remember the golden rule: `fit_transform` on train, `transform` only on test.

In [8]:
# Binary columns mapping
binary_cols = ["Partner", "Dependents", "PhoneService", "PaperlessBilling"]
for c in binary_cols:
    X_train[c] = X_train[c].map({"Yes": 1, "No": 0})
    X_test[c] = X_test[c].map({"Yes": 1, "No": 0})

# One-Hot Encoding for categorical columns
cat_cols = ["gender", "MultipleLines", "InternetService", "OnlineSecurity",
            "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV",
            "StreamingMovies", "Contract", "PaymentMethod"]

X_train = pd.get_dummies(X_train, columns=cat_cols)
X_test = pd.get_dummies(X_test, columns=cat_cols)

# Ensure X_test has the exact same columns as X_train
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

# Scale numeric columns
num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

In [9]:
# Check your progress
non_numeric = X_train.select_dtypes(exclude=[np.number, bool]).columns.tolist()
assert len(non_numeric) == 0, f"These columns are still non-numeric, encoding is incomplete: {non_numeric}"
assert list(X_train.columns) == list(X_test.columns), "X_train and X_test have different columns -- check your reindex step."
assert abs(X_train["tenure"].mean()) < 0.1, "tenure doesn't look scaled (mean should be ~0) -- did you fit_transform the scaler on X_train?"
print("Section 5 looks good -- all columns numeric, train/test aligned, scaling applied.")


Section 5 looks good -- all columns numeric, train/test aligned, scaling applied.


<details>
<summary>Solution</summary>

```python
binary_cols = ["Partner", "Dependents", "PhoneService", "PaperlessBilling"]
for c in binary_cols:
    X_train[c] = X_train[c].map({"Yes": 1, "No": 0})
    X_test[c] = X_test[c].map({"Yes": 1, "No": 0})

cat_cols = ["gender", "MultipleLines", "InternetService", "OnlineSecurity",
            "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV",
            "StreamingMovies", "Contract", "PaymentMethod"]
X_train = pd.get_dummies(X_train, columns=cat_cols)
X_test = pd.get_dummies(X_test, columns=cat_cols)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])
```
</details>


## 6. The Leakage Experiment

This is the heart of the capstone. Leakage doesn't always look the same sometimes it's a subtle statistical nudge, sometimes it's catastrophic. You'll measure both.

### 6A. The Subtle Case

**Feature:** `PaymentMethod_ChurnRate` the historical churn rate for each customer's payment method (a form of *target encoding*). This is a genuinely useful feature in the real world... if computed correctly.

- **Version A (leaky):** compute each payment method's churn rate using the **entire dataset** (train + test combined), *before* splitting.
- **Version B (correct):** compute it using **training data only**, then map those same rates onto the test set.

Build both, then compare the actual rate values side by side.

In [11]:

# 1 Leaky Version: Compute churn rate using the FULL dataset before splitting
leaky_rates = df.groupby("PaymentMethod")["Churn"].mean()

# 2 Correct Version: Compute churn rate using ONLY the training rows (before encoding changed the original columns)
train_only_original = df.loc[X_train.index]
correct_rates = train_only_original.groupby("PaymentMethod")["Churn"].mean()

# 3 Compare both results in a dataframe
comparison = pd.DataFrame({
    "leaky (full data)": leaky_rates,
    "correct (train only)": correct_rates
})

# 4 Calculate the absolute difference to see how much leakage changed the values
comparison["abs_difference"] = (comparison["leaky (full data)"] - comparison["correct (train only)"]).abs()

# Display the comparison table
comparison

,leaky (full data),correct (train only),abs_difference
PaymentMethod,,,
Bank transfer (automatic),0.167315,0.169687,0.002372
Credit card (automatic),0.152531,0.149023,0.003508
Electronic check,0.451462,0.456085,0.004622
Mailed check,0.190176,0.184169,0.006007


### Read the numbers honestly

With `PaymentMethod` (only 4 categories spread across 5,600+ training rows), the leaked and correct rates will likely differ by **only a few percentage points** and a downstream model's train/test accuracy gap may barely move. That's a real, honest result, not a failed experiment.

**This is itself the lesson:** leakage severity scales with *how few rows sit behind each group*. Here, each payment method is backed by over a thousand rows, so leaking a handful of test rows into the average barely shifts it.

Now recall **`SellerG`** from Activity2 330 sellers spread across ~15,000 rows, some with only a handful of listings each. Compute the same style of comparison there in your head: with ~45 rows per group instead of ~1,400, the exact same mistake would shift individual group averages dramatically, not by a few points.

**`# TODO`** Answer in the markdown cell below: given what you just measured, would you be *more* or *less* worried about this same leak on a categorical feature with 300+ categories instead of 4? Why?

*(I would be more worried

With 4 categories, each group has over a thousand rows so leaking a few test rows barely shifts the average

With 300+ categories many groups will have very few rows (like just 1 or 2) meaning leaking test data would dramatically distort the group averages and cause severe data leakage)*


<details>
<summary>Solution 6A</summary>

```python
# LEAKY: computed on the full dataset before splitting
leaky_rates = df.groupby("PaymentMethod")["Churn"].mean()

# CORRECT: computed on training rows only, after splitting
train_only = df.loc[X_train.index]
correct_rates = train_only.groupby("PaymentMethod")["Churn"].mean()

comparison = pd.DataFrame({
    "leaky (full data)": leaky_rates,
    "correct (train only)": correct_rates,
})
comparison["abs_difference"] = (comparison["leaky (full data)"] - comparison["correct (train only)"]).abs()
print(comparison)
```

**Why the gap is small here:** `PaymentMethod` has only 4 categories spread across 5,600+ training rows (roughly 1,400 rows backing each average). Leaking a few hundred test rows into that average barely moves it the law of large numbers is doing you a favor, even though the *methodology* is still wrong.

**Why it wouldn't stay small:** Go back to `SellerG` in Activity2 330 categories across ~15,000 rows, ~45 rows per group. The exact same leak, on a feature with that few rows per category, would shift individual group averages substantially, because a handful of leaked test rows are a much bigger share of a small group. Same mistake, very different consequences driven entirely by cardinality and sample size, not by how careful you were.

**The takeaway:** don't judge whether a leak is "safe" by whether today's dataset happens to hide its effect. Fix the methodology every time, regardless of the number you see in front of you.
</details>


### 6B. The Catastrophic Case (synthetic, for illustration)

Not every leak is subtle. The most dangerous ones happen when a feature is only ever populated **after** the outcome you're trying to predict has already occurred — a form of *temporal leakage*.

This next column doesn't exist in the real dataset — we're building it here purely to make the failure mode vivid: imagine an operations team opens a **"retention case"** for a customer, but only *after* that customer has already churned. If you didn't know that business rule and just saw a column called `retention_case_opened` sitting in your table, it looks like an innocent feature.

In [12]:
# SYNTHETIC DEMO ONLY -- this field does not exist in the real dataset.
# In reality, a "retention case" would only be opened AFTER a customer churns --
# so at prediction time (before churn happens), this value could never actually be known.
df_demo = df.copy()
df_demo["retention_case_opened"] = df_demo["Churn"]  # <-- directly derived from the label itself

demo_features = ["tenure", "MonthlyCharges", "retention_case_opened"]
Xd_train, Xd_test, yd_train, yd_test = train_test_split(
    df_demo[demo_features], df_demo["Churn"], test_size=0.2, random_state=42, stratify=df_demo["Churn"]
)

clf_leak = LogisticRegression(max_iter=1000)
clf_leak.fit(Xd_train, yd_train)
print("WITH catastrophic leak  -- train acc:", accuracy_score(yd_train, clf_leak.predict(Xd_train)),
      " test acc:", accuracy_score(yd_test, clf_leak.predict(Xd_test)))

Xd_train2 = Xd_train.drop(columns=["retention_case_opened"])
Xd_test2 = Xd_test.drop(columns=["retention_case_opened"])
clf_clean = LogisticRegression(max_iter=1000)
clf_clean.fit(Xd_train2, yd_train)
print("WITHOUT the leak feature -- train acc:", accuracy_score(yd_train, clf_clean.predict(Xd_train2)),
      " test acc:", accuracy_score(yd_test, clf_clean.predict(Xd_test2)))


WITH catastrophic leak  -- train acc: 1.0  test acc: 1.0
WITHOUT the leak feature -- train acc: 0.7860199714693296  test acc: 0.7838801711840229


### Notice what changed

Unlike 6A, this isn't a train/test **gap** both train *and* test accuracy jump to essentially 100%. That's actually a second, distinct red flag worth remembering:

- **6A's signature:** a gap between train and test performance
- **6B's signature:** suspiciously *perfect* performance on both nothing this good survives contact with production

If a model ever looks "too good to be true" in either sense, that's exactly when to go looking for a feature that's secretly derived from or only available after the outcome you're predicting.


## 7. Final Model

Using your **correct, non-leaky** feature set from Sections 3-5 (the catastrophic `retention_case_opened` demo in 6B was synthetic and illustrative only don't carry it into your real pipeline), train one more Logistic Regression and report:
- Accuracy
- F1-score
- Confusion matrix

This is your finish line a clean, honest pipeline, not necessarily the highest possible score.

In [13]:
# Train the final Logistic Regression model using clean and non-leaky data
final_clf = LogisticRegression(max_iter=1000)
final_clf.fit(X_train, y_train)

# Make predictions on the test set
final_preds = final_clf.predict(X_test)

# Calculate the metrics
acc = accuracy_score(y_test, final_preds)
f1 = f1_score(y_test, final_preds)
conf_matrix = confusion_matrix(y_test, final_preds)

# Print out the results clearly
print(f"Accuracy: {acc:.4f}")
print(f"F1-score: {f1:.4f}")
print("Confusion Matrix:")
print(conf_matrix)


Accuracy: 0.8074
F1-score: 0.5970
Confusion Matrix:
[[932  99]
 [171 200]]


In [14]:
# Check your progress
final_preds = final_clf.predict(X_test) if "final_clf" in dir() else clf.predict(X_test)
acc = accuracy_score(y_test, final_preds)
assert 0.65 < acc < 0.95, f"Accuracy of {acc:.3f} is outside the expected range for this dataset -- double check your pipeline."
print(f"Final test accuracy: {acc:.3f}  ✅ in the expected range for this dataset")


Final test accuracy: 0.807  ✅ in the expected range for this dataset


<details>
<summary>Solution</summary>

```python
final_clf = LogisticRegression(max_iter=2000)
final_clf.fit(X_train, y_train)

final_preds = final_clf.predict(X_test)
print("Accuracy:", accuracy_score(y_test, final_preds))
print("F1:", f1_score(y_test, final_preds))
print(confusion_matrix(y_test, final_preds))
print(classification_report(y_test, final_preds))
```
</details>


## 8. Reflection

1. What did `isnull()` miss in `TotalCharges`, and why?
2. Why did we use `stratify=y` for this dataset specifically, when we didn't need it in Activity 2?
3. In your `PaymentMethod` experiment (6A), was the leaky/correct gap large or small? Why and what would change if the feature had 300+ categories instead of 4?
4. In 6B, the giveaway wasn't a train/test gap it was suspiciously perfect scores on *both*. Why does that pattern specifically point to leakage rather than "a great model"?
5. Map this notebook back to the lecture's **Best Practices Checklist** which item did Section 6 make concrete for you?

*(1- missed empty strings like spaces " " stored as text because isnull() only checks for NaN or None values not hidden whitespace strings

2_ Because this dataset is imbalanced (churn is rare) so stratify=y ensures both train and test sets keep the exact same proportion of positive and negative classes as the original data

3- The gap was small because PaymentMethod has only 4 categories each backed by over a thousand rows so leaking a few test rows barely shifted the average With 300+ categories many groups it have very few rows like 1 and 2 so leaking test data would dramatically distort group averages and cause severe leakage

4- Because real-world data is inherently noisy and complex nothing is ever perfect A suspiciously perfect score means the model is cheating by using a feature that is directly derived from or only available after the outcome temporal/target leakage

5-  Section 6 made the Never touch the test data during feature engineering checklist item real and clear It showed me practically why we must always split our data first, and only calculate things like averages or encodings on the training set to avoid cheating )*


---
### Dataset credit

Telco Customer Churn dataset, originally published by **IBM** as a sample dataset for customer retention analytics. Used here for educational purposes.
